<a href="https://colab.research.google.com/github/anokhina-rgb/Consecutive-Translation-Trainer/blob/main/10sec%D1%96.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🛠️ Інструкція з Використання (ВЕРСІЯ 8)
Налаштуйте CPU (для стабільності):

У меню Colab: Середовище виконання (Runtime) → Змінити тип середовища виконання (Change runtime type).

У полі Прискорювач (Hardware accelerator) виберіть None (це CPU).

Натисніть Зберегти.
(Цей крок необхідний, щоб запобігти відключенню через обмеження GPU.)

Запустіть Код:

Натисніть кнопку "Play" (трикутник) біля першої комірки коду.

Завантажте Аудіо:

Коли з'явиться вікно, завантажте ваш аудіофайл (.mp3 або .wav).

Увага: Процес на CPU буде тривалим (кілька хвилин), не закривайте вікно.

Результат:

Після завершення ви побачите повідомлення: "✅ Роздаточний матеріал готовий. Завантажте файл material_for_students.zip."

Файл material_for_students.zip автоматично завантажиться на ваш комп'ютер.

# %% [markdown]
# **Атрибуція:** Анохіна Тетяна © 2025. Код ліцензовано CC BY 4.0.
#
# **Вихідні бібліотеки, що використовуються:**
# * Whisper (OpenAI)
# * Pydub
# * Python-docx
#
# ---

In [ ]:
# %% [markdown]
# # 🎧 Consecutive Translation Trainer - ВЕРСІЯ 7 (ОСТАТОЧНА СТАБІЛЬНІСТЬ)
#
# **ПРИНЦИП:** Максимальна надійність, навіть якщо за рахунок ідеальної точності. Використовує тільки грубі сегменти Whisper.

# %%
# Install all required dependencies
!pip install -q openai-whisper pydub matplotlib python-docx googletrans==4.0.0rc1
import os
import re
import shutil
import numpy as np
import whisper
from pydub import AudioSegment
import matplotlib.pyplot as plt
from docx import Document
import zipfile
from google.colab import files
from googletrans import Translator
import time
import torch

# --- Configuration ---
WHISPER_MODEL = "medium"
MINIMUM_PAUSE_SECONDS = 10
SENTENCE_ENDINGS = ['.', '?', '!']
AUDIO_END_BUFFER_MS = 500

# DYNAMIC DEVICE SELECTION (Automatic Fallback)
if torch.cuda.is_available():
    device = "cuda"
    print("🚀 Виявлено GPU. Використання прискорювача CUDA.")
else:
    device = "cpu"
    print("🐌 GPU не виявлено. Використання CPU. Перевірте налаштування Colab Runtime.")


# --- Налаштування робочих папок ---
input_folder = "/content/input_audio"
output_folder = "/content/output_material"
temp_folder = "/content/temp_files"

if os.path.exists(input_folder): shutil.rmtree(input_folder)
if os.path.exists(output_folder): shutil.rmtree(output_folder)
if os.path.exists(temp_folder): shutil.rmtree(temp_folder)
for f in os.listdir('/content'):
    if f.endswith(('.png', '.docx', '.mp3', '.zip')):
        try: os.remove(os.path.join('/content', f))
        except OSError: pass

os.makedirs(input_folder)
os.makedirs(output_folder)
os.makedirs(temp_folder)

# --- 1. Завантаження аудіофайлу ---
print("Завантажте ваш аудіофайл (mp3/wav):")
uploaded = files.upload()

if not uploaded:
    print("❌ Файл не завантажено. Припинення роботи.")
    exit()

audio_original_filename = list(uploaded.keys())[0]
audio_processing_path = os.path.join(input_folder, audio_original_filename)
os.rename(audio_original_filename, audio_processing_path)
audio = AudioSegment.from_file(audio_processing_path)

# --- 2. Транскрипція та поділ на ГРУБІ сегменти Whisper ---
start_time = time.time()
print(f"Завантаження моделі '{WHISPER_MODEL}' на пристрої: {device}")
model_loaded = whisper.load_model(WHISPER_MODEL, device=device)

print("Виконується транскрипція (ОДИН РАЗ, для максимальної надійності)...")
# Виконуємо транскрипцію лише один раз
result = model_loaded.transcribe(audio_processing_path)
full_text = result["text"]

# Тепер ми використовуємо *грубі* сегменти, які Whisper створив сам
semantic_segments = result.get('segments', [])

# Корекція тексту для документації
corrected_text = full_text

print(f"Час транскрипції: {time.time() - start_time:.2f}с")
print(f"✅ Поділ сегментів завершено. Згенеровано {len(semantic_segments)} навчальних сегментів.")


# =========================================================
# --- 3. Обробка аудіо та додавання пауз (якщо є сегменти) ---
# =========================================================
output_audio = AudioSegment.silent(duration=0)
timeline = []
pause_duration_ms = MINIMUM_PAUSE_SECONDS * 1000

for i, segment in enumerate(semantic_segments):
    start_time = int(segment['start'] * 1000)
    end_time_raw = int(segment['end'] * 1000)
    end_time_buffered = end_time_raw + AUDIO_END_BUFFER_MS
    end_time = min(end_time_buffered, len(audio))
    text = segment['text'].strip()

    if not text or start_time >= end_time:
        continue

    segment_audio = audio[start_time:end_time]
    output_audio += segment_audio
    output_audio += AudioSegment.silent(duration=pause_duration_ms)

    timeline_entry = f"Chunk {i+1} ({len(text.split())} слів, {segment['end'] - segment['start']:.2f}с): {start_time/1000:.2f}с -> {end_time_raw/1000:.2f}с (+{AUDIO_END_BUFFER_MS}ms buffer) | {text}"
    timeline.append(timeline_entry)

print(f"✅ Обробка аудіо та створення пауз завершено. Тривалість паузи: {MINIMUM_PAUSE_SECONDS}с.")


# --- 4. Генерація файлів (ТОЧНИЙ МАКЕТ DOCX) ---
def create_waveform_image(audio_segment, filename):
    data = np.array(audio_segment.get_array_of_samples())
    channels = audio_segment.channels
    sample_rate = audio_segment.frame_rate
    times = np.linspace(0, len(data) / sample_rate / channels, num=len(data) // channels)
    plt.figure(figsize=(15, 5))
    plt.plot(times, data[::channels])
    plt.title(filename)
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.savefig(os.path.join(temp_folder, filename))
    plt.close()

waveform_orig_filename = "waveform_original.png"
waveform_pauses_filename = "waveform_with_pauses.png"
doc_filename = "handout_final.docx"
audio_orig_filename = "original.mp3"
audio_pauses_filename = "with_pauses_final.mp3"
zip_filename = "material_for_students.zip"

create_waveform_image(audio, waveform_orig_filename)
create_waveform_image(output_audio, waveform_pauses_filename)

doc = Document()
doc.add_heading('Сторінка 1: Оригінальна транскрипція', level=2)
doc.add_paragraph(corrected_text)
doc.add_picture(os.path.join(temp_folder, waveform_orig_filename))
doc.add_page_break()
doc.add_heading('Сторінка 2: Точні таймінги та Аудіо з паузами', level=2)
doc.add_paragraph('**Точні таймінги (Consecutive Segments):**')
doc.add_paragraph('\n'.join(timeline))
doc.add_picture(os.path.join(temp_folder, waveform_pauses_filename))
doc.add_page_break()

# --- Переклад (ЗАХИСТ ВІД ЗБОЮ МЕРЕЖІ) ---
doc.add_heading("Сторінка 3: Український переклад", level=2)
translator = Translator()
full_text_from_segments = "\n\n".join([s['text'] for s in semantic_segments])
try:
    ukrainian_translation = translator.translate(full_text_from_segments, dest='uk').text
except Exception as e:
    print("❌ Помилка перекладу: Не вдалося підключитися до API Google Translate.")
    ukrainian_translation = (
        f"Помилка перекладу: Не вдалося підключитися до API. Спробуйте вручну перекласти текст нижче:\n\n"
        f"ТЕКСТ ДЛЯ ПЕРЕКЛАДУ:\n\n{full_text_from_segments}"
    )

doc.add_paragraph(ukrainian_translation)
doc.save(os.path.join(output_folder, doc_filename))

output_audio.export(os.path.join(output_folder, audio_pauses_filename), format="mp3")
audio.export(os.path.join(output_folder, audio_orig_filename), format="mp3")

# --- 5. Пакетування та завантаження ---
final_zip_path = os.path.join("/content", zip_filename)

if os.path.exists(os.path.join(output_folder, doc_filename)):
    with zipfile.ZipFile(final_zip_path, "w") as zf:
        zf.write(os.path.join(output_folder, audio_orig_filename), audio_orig_filename)
        zf.write(os.path.join(output_folder, audio_pauses_filename), audio_pauses_filename)
        zf.write(os.path.join(output_folder, doc_filename), doc_filename)
        if os.path.exists(os.path.join(temp_folder, waveform_orig_filename)):
          zf.write(os.path.join(temp_folder, waveform_orig_filename), waveform_orig_filename)
        if os.path.exists(os.path.join(temp_folder, waveform_pauses_filename)):
          zf.write(os.path.join(temp_folder, waveform_pauses_filename), waveform_pauses_filename)


    print("✅ Роздаточний матеріал готовий. Завантажте файл **material_for_students.zip**.")
    files.download(final_zip_path)
else:
    print("❌ Обробка не вдалася. ZIP-файл не створено.")